# Download S&P 500 and S&P 600 Tickers (Full Lists)

Fetches **full** constituent lists and appends only new stock tickers to `research/raw/stocks.csv`. The file contains both **ETFs and stocks**; existing rows (including all ETFs) are kept without duplication.

- **S&P 500**: [GitHub CSV](https://github.com/datasets/s-and-p-500-companies) (~500 tickers, no API key).
- **S&P 600**: Wikipedia list of S&P 600 companies (~600 tickers).

Columns written: **Sector**, **Ticker**, **Full Name**, **Short Description**. The notebook reports how many new tickers were added from each index.

In [9]:
import sys
from pathlib import Path

import pandas as pd

_root = Path.cwd().resolve()
while _root != _root.parent and not (_root / ".git").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))

from research.functions.download_helper import find_project_root

PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "research" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

In [10]:
SP500_CSV_URL = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/master/data/constituents.csv"
SP600_WIKI_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_600_companies"


def fetch_sp500() -> pd.DataFrame:
    """Full S&P 500 constituents from GitHub CSV (Symbol, Security, GICS Sector, GICS Sub-Industry)."""
    df = pd.read_csv(SP500_CSV_URL)
    sub_col = "GICS Sub-Industry" if "GICS Sub-Industry" in df.columns else next(
        (c for c in df.columns if "Sub" in c and "Industry" in c), None
    )
    desc = df[sub_col].astype(str).str.strip() if sub_col else pd.Series([""] * len(df), index=df.index)
    out = pd.DataFrame({
        "Sector": df["GICS Sector"].astype(str).str.strip(),
        "Ticker": df["Symbol"].astype(str).str.strip(),
        "Full Name": df["Security"].astype(str).str.strip(),
        "Short Description": desc,
    })
    return out.dropna(subset=["Ticker"]).drop_duplicates(subset=["Ticker"], keep="first")


def fetch_sp600() -> pd.DataFrame:
    """Full S&P 600 constituents from Wikipedia table. Uses a browser User-Agent to avoid 403."""
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; rv:109.0) Gecko/20100101 Firefox/115.0"}
    tables = pd.read_html(SP600_WIKI_URL, storage_options=headers)
    df = tables[0].copy()
    df.columns = [c.strip() if isinstance(c, str) else "_".join(str(x) for x in c).strip() for c in df.columns]
    sym_col = next((c for c in df.columns if "Symbol" in c or c == "Ticker"), df.columns[0])
    sec_col = next((c for c in df.columns if "GICS" in c and "Sector" in c), df.columns[2] if len(df.columns) > 2 else "Sector")
    name_col = next((c for c in df.columns if "Security" in c), df.columns[1] if len(df.columns) > 1 else "Security")
    sub_col = next((c for c in df.columns if "Sub-Industry" in c or "Sub Industry" in c), "")
    if not sub_col:
        sub_col = next((c for c in df.columns if "Industry" in c), "")
    desc = df[sub_col].astype(str).str.strip() if sub_col and sub_col in df.columns else pd.Series([""] * len(df), index=df.index)
    out = pd.DataFrame({
        "Sector": df[sec_col].astype(str).str.strip(),
        "Ticker": df[sym_col].astype(str).str.strip(),
        "Full Name": df[name_col].astype(str).str.strip(),
        "Short Description": desc,
    })
    return out.dropna(subset=["Ticker"]).drop_duplicates(subset=["Ticker"], keep="first")

In [11]:
sp500 = fetch_sp500()
sp600 = fetch_sp600()
print(f"S&P 500: {len(sp500)} tickers")
print(f"S&P 600: {len(sp600)} tickers")

S&P 500: 503 tickers
S&P 600: 603 tickers


In [12]:
STOCKS_CSV = RAW_DIR / "stocks.csv"
COLS = ["Sector", "Ticker", "Full Name", "Short Description"]

# Load existing universe (ETFs + stocks; same columns). Drop rows where Ticker looks numeric (bad data from prior runs).
if STOCKS_CSV.exists():
    existing = pd.read_csv(STOCKS_CSV)
    existing = existing[existing["Ticker"].notna()].astype({c: str for c in COLS})
    existing["Ticker"] = existing["Ticker"].str.strip()
    existing = existing[~existing["Ticker"].str.match(r"^[\d.]+$", na=False)]  # drop numeric "tickers"
else:
    existing = pd.DataFrame(columns=COLS)

existing_tickers = set(existing["Ticker"].str.strip())

# New tickers from S&P 500 (not already in stocks)
new_sp500 = sp500[~sp500["Ticker"].str.strip().isin(existing_tickers)].copy()
existing_tickers |= set(new_sp500["Ticker"].str.strip())

# New tickers from S&P 600 (not already in stocks or from sp500)
new_sp600 = sp600[~sp600["Ticker"].str.strip().isin(existing_tickers)].copy()

# Combine: existing first, then new from sp500, then new from sp600 (no duplicates)
updated = pd.concat([existing, new_sp500, new_sp600], ignore_index=True)
updated.to_csv(STOCKS_CSV, index=False)
print(f"Wrote {STOCKS_CSV} ({len(updated)} total rows)")

Wrote /Users/mdabdullahalmahin/Desktop/Projects/quant-trading/research/raw/stocks.csv (1101 total rows)


In [13]:
# Report: new tickers added
print("--- New tickers added to stocks.csv ---")
print(f"From S&P 500: {len(new_sp500)}")
print(f"From S&P 600: {len(new_sp600)}")
print(f"Total new:    {len(new_sp500) + len(new_sp600)}")

# Preview
display(sp500.head(10))
display(sp600.head(10))

--- New tickers added to stocks.csv ---
From S&P 500: 361
From S&P 600: 595
Total new:    956


,Sector,Ticker,Full Name,Short Description
0,Industrials,MMM,3M,Industrial Conglomerates
1,Industrials,AOS,A. O. Smith,Building Products
2,Health Care,ABT,Abbott Laboratories,Health Care Equipment
3,Health Care,ABBV,AbbVie,Biotechnology
4,Information Technology,ACN,Accenture,IT Consulting & Other Services
5,Information Technology,ADBE,Adobe Inc.,Application Software
6,Information Technology,AMD,Advanced Micro Devices,Semiconductors
7,Utilities,AES,AES Corporation,Independent Power Producers & Energy Traders
8,Financials,AFL,Aflac,Life & Health Insurance
9,Health Care,A,Agilent Technologies,Life Sciences Tools & Services


,Sector,Ticker,Full Name,Short Description
0,Financials,AAMI,Acadian Asset Management Inc.,Asset Management & Custody Banks
1,Consumer Discretionary,AAP,"Advance Auto Parts, Inc.",Automotive Retail
2,Real Estate,AAT,American Assets Trust,Diversified REITs
3,Financials,ABCB,Ameris Bancorp,Regional Banks
4,Consumer Discretionary,ABG,Asbury Automotive Group,Automotive Retail
5,Industrials,ABM,"ABM Industries, Inc.",Environmental & Facilities Services
6,Financials,ABR,Arbor Realty Trust,Mortgage REITs
7,Industrials,ACA,"Arcosa, Inc.",Construction & Engineering
8,Health Care,ACAD,Acadia Pharmaceuticals,Pharmaceuticals
9,Health Care,ACHC,Acadia Healthcare,Health Care Facilities
